# IND320 Project Work, Part 3

**Student:** Sara Christine Gjerløw  

**GitHub repository:** https://github.com/SaraGjerlow/ind320-SaraGjerlow  
**Streamlit app:** https://ind320-saragjerlow.streamlit.app

This part adds weather data. I sample locations inside the NVE areas, download
historical weather from open-meteo for those locations, and look for outliers
and patterns in precipitation, temperature and reservoir filling.

## 1. Imports & database connection

In [ ]:
import tomllib
from pathlib import Path
import time

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi
from scipy.fft import dct, idct
from scipy.signal import spectrogram
from sklearn.neighbors import LocalOutlierFactor
from statsmodels.tsa.seasonal import STL
import requests_cache

In [ ]:
# The MongoDB connection string is kept out of GitHub in secrets.toml.
with open(Path("../.streamlit/secrets.toml"), "rb") as f:
    secrets = tomllib.load(f)

mongo_uri = secrets["mongo"]["uri"]

# Check the connection before using it later.
mongo_client = MongoClient(mongo_uri, server_api=ServerApi("1"))

try:
    mongo_client.admin.command("ping")
    print("Connected to MongoDB.")
except Exception as error:
    print(error)

In [ ]:
# Responses are stored on disk and never expire, so a batch that has already
# been downloaded is never requested again. This makes the download
# resumable: if the API quota runs out, the next run picks up where this one
# stopped, and later runs of the whole notebook cost nothing.
session = requests_cache.CachedSession(".openmeteo_cache", expire_after=-1)

## 2. Sampling locations from the NVE shapefiles

The weather data has to be downloaded for coordinates, but Magasinstatistikk
reports by area. NVE publishes the area borders as shapefiles, so I draw 1000
random points inside each area and use those as download locations.

NVE operates with five regions, while Magasinstatistikk reports three
watercourse areas. Region Sør and Region Øst together make up the south-east
area, and Region Midt and Region Nord together make up the northern one. This
is also why VASS 4 never shows up in the data, which I found in part 2.

In [ ]:
SHAPEFILE_DIR = Path("../shapefiles")
POINTS_PER_AREA = 1000
RANDOM_STATE = 42

# NVE's five regions grouped into the three watercourse areas that
# Magasinstatistikk actually reports.
REGION_TO_VASS = {"RS": 1, "RØ": 1, "RV": 2, "RM": 3, "RN": 3}


def sample_points(polygons, area_type):
    """Draw random points inside each polygon and return them as a table.

    The shapefiles use UTM coordinates in metres, so the points are converted
    to ordinary latitude and longitude, which is what open-meteo expects.
    """
    points = polygons["geometry"].sample_points(
        size=POINTS_PER_AREA, rng=RANDOM_STATE
    )
    points = points.to_crs(epsg=4326)

    rows = []
    for index, multipoint in points.items():
        area_number = polygons.loc[index, "area_number"]
        for point in multipoint.geoms:
            rows.append({
                "area_type": area_type,
                "area_number": int(area_number),
                "latitude": round(point.y, 4),
                "longitude": round(point.x, 4),
            })

    return pd.DataFrame(rows)


# The electricity price areas are named "NO 1" to "NO 5" in the file.
# The encoding has to be given, or Norwegian letters come out wrong.
el_areas = gpd.read_file(
    SHAPEFILE_DIR / "Budområder/Budområder.shp", encoding="utf-8"
)
el_areas["area_number"] = el_areas["budomr"].str.replace("NO ", "").astype(int)
el_points = sample_points(el_areas, "EL")

# The regions are merged two and two before sampling, so the points are spread
# over the whole area instead of 1000 per NVE region.
vass_areas = gpd.read_file(
    SHAPEFILE_DIR / "NVE_Region/NVE_Region.shp", encoding="utf-8"
)
vass_areas["area_number"] = vass_areas["REGION"].map(REGION_TO_VASS)
vass_areas = vass_areas.dissolve(by="area_number", as_index=False)
vass_points = sample_points(vass_areas, "VASS")

points = pd.concat([el_points, vass_points], ignore_index=True)

print("Points per area:")
print(points.groupby(["area_type", "area_number"]).size())
points.head()

## 3. Store the sampled points in MongoDB

The Streamlit app needs the same coordinates, but it should not have to read
the shapefiles. That would mean installing geopandas on Streamlit Cloud, which
is heavy and slow to build. So the points are stored in MongoDB here, and the
app reads them from there before asking open-meteo for the weather.

In [ ]:
database = mongo_client["ind320"]
collection = database["area_points"]

# Emptied first, so running the notebook again does not duplicate anything.
collection.delete_many({})

# MongoDB stores dictionaries. to_dict("records") makes one per row.
collection.insert_many(points.to_dict("records"))

print("Documents in collection:", collection.count_documents({}))

## 4. Weather data from open-meteo

The function below takes an area, looks up its coordinates in MongoDB, and asks
open-meteo for hourly historical weather from the ERA5 reanalysis model.

A thousand locations is too much for one request, so they are sent in batches.
The function does not keep every single series. It adds them up as they arrive
and divides at the end, so the result is one average row per hour for the whole
area. That is what the plots further down need.

The analyses below use 100 of the 1000 sampled locations for one area.
Open-meteo limits how much can be downloaded per day, so the full set has to
be collected over several days. The cached responses mean nothing is lost
between runs, and raising the number is the only change needed once it is
complete.

In [ ]:
ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"

HOURLY_VARIABLES = [
    "temperature_2m",
    "precipitation",
    "snowfall",
    "relative_humidity_2m",
    "wind_speed_10m",
]

LOCATIONS_PER_REQUEST = 25


def load_area_points(area_type, area_number):
    """Read the sampled coordinates for one area back from MongoDB."""
    documents = collection.find(
        {"area_type": area_type, "area_number": area_number},
        {"_id": 0, "latitude": 1, "longitude": 1},
    )
    return pd.DataFrame(list(documents))


def fetch_batch(parameters, pause_seconds=65):
    """Ask open-meteo for one batch of locations.

    Open-meteo limits how much data can be asked for per minute, per hour and
    per day. A rejected request is not an error here, it just means waiting.
    Cached responses never reach the API at all, so a run that stopped on a
    limit continues where it left off the next time.
    """
    for attempt in range(6):
        response = session.get(ARCHIVE_URL, params=parameters, timeout=180)

        if response.status_code == 200:
            return response.json()

        if response.status_code == 429:
            # The body says which of the three limits was reached.
            reason = response.json().get("reason", "rate limited")
            print(f"    {reason}")
            time.sleep(pause_seconds)
            continue

        # Anything else is a real error worth seeing.
        print("Status:", response.status_code)
        print(response.text[:300])
        response.raise_for_status()

    raise RuntimeError("Gave up after repeated rate limits")


def download_weather(area_type, area_number, start_date, end_date,
                     max_locations=None,
                     locations_per_request=LOCATIONS_PER_REQUEST):
    """Download hourly ERA5 weather for one area and average over its points.

    The area is given with the same type and number that Magasinstatistikk
    uses, for example "EL" and 3. The coordinates are looked up in MongoDB,
    sent to open-meteo in batches, and averaged into one series per variable.
    """
    coordinates = load_area_points(area_type, area_number)

    if max_locations is not None:
        coordinates = coordinates.head(max_locations)

    print(f"Downloading {len(coordinates)} locations for {area_type} {area_number}")

    total = None
    count = 0

    for start in range(0, len(coordinates), locations_per_request):
        batch = coordinates.iloc[start:start + locations_per_request]

        parameters = {
            "latitude": ",".join(batch["latitude"].astype(str)),
            "longitude": ",".join(batch["longitude"].astype(str)),
            "start_date": start_date,
            "end_date": end_date,
            "hourly": ",".join(HOURLY_VARIABLES),
            "models": "era5",
        }

        results = fetch_batch(parameters)

        # One location gives a single object back, several give a list.
        if isinstance(results, dict):
            results = [results]

        for result in results:
            hourly = pd.DataFrame(result["hourly"])
            hourly["time"] = pd.to_datetime(hourly["time"])
            hourly = hourly.set_index("time")

            if total is None:
                total = hourly
            else:
                total = total + hourly

            count += 1

        print(f"  {count} of {len(coordinates)} locations done")

    weather = total / count
    return weather.reset_index()

In [ ]:
# All the analyses below use this one area and this one year.
#
# The goal is 1000 locations, but open-meteo has a daily quota, so the
# download has to be spread over several days. The cache keeps everything
# that has already arrived, so re-running this cell on a new day moves it
# further along. Raise the number here when the full set is in place.
LOCATIONS = 100

weather_no3 = download_weather(
    "EL", 3, "2024-01-01", "2024-12-31", max_locations=LOCATIONS
)

print("Hours:", len(weather_no3))
print("Locations averaged:", LOCATIONS)
weather_no3.head()

The Streamlit app needs the same weather, but it should not download it again
on every rerun. That would be slow and would spend the daily open-meteo limit
for no reason. So the finished average is stored in MongoDB here, the same way
the sampled coordinates are, and the app only reads it.

In [ ]:
weather_collection = database["area_weather"]

# Emptied for this area first, so running the notebook again does not
# duplicate anything.
weather_collection.delete_many({"area_type": "EL", "area_number": 3})

# MongoDB stores dictionaries, so the area is written into every row.
documents = []
for row in weather_no3.to_dict("records"):
    row["area_type"] = "EL"
    row["area_number"] = 3
    documents.append(row)

weather_collection.insert_many(documents)

print("Hours stored:", weather_collection.count_documents(
    {"area_type": "EL", "area_number": 3}
))

## 5. Outliers in precipitation (DCT and SPC)

Precipitation has a slow seasonal pattern and fast day-to-day variation on top
of it. To find unusual weather I have to separate the two, because a rainy day
in October is not unusual while the same amount in a dry period might be.

The Discrete Cosine Transform splits the series into frequencies. Keeping only
the lowest ones gives the slow seasonal shape, and subtracting that leaves the
fast variation, here called SATV. The outlier limits are calculated from SATV
and then drawn back onto the original curve, so the boundaries follow the
season instead of being two straight lines.

The limits use the median and the median absolute deviation rather than mean
and standard deviation. A few very wet hours would inflate an ordinary
standard deviation enough to hide themselves inside the limits.

The limits are symmetric around the seasonal level, so the lower one ends up
below zero. Precipitation cannot be negative, so in practice only the upper
limit ever flags anything. I have kept both to show the method as it is.

In [ ]:
def precipitation_outliers(weather, cutoff=25, n_sigma=3.0,
                           column="precipitation"):
    """Plot precipitation with SPC limits and mark the outliers.

    cutoff is how many low DCT frequencies count as seasonal. A low number
    keeps only the slowest shape, a high number follows the curve more closely.

    n_sigma is how many robust standard deviations away from the seasonal
    level a point has to be before it counts as an outlier.

    Returns the figure and a table of the outliers.
    """
    series = weather.set_index("time")[column]
    values = series.to_numpy()

    # Split into frequencies, keep only the slowest ones, and transform back.
    # What is left is the seasonal shape of the year.
    coefficients = dct(values, norm="ortho")
    coefficients[cutoff:] = 0.0
    seasonal = idct(coefficients, norm="ortho")

    # SATV is what remains when the season is removed. It is only used to find
    # the limits, it is never plotted.
    satv = values - seasonal

    # Robust statistics, so that the outliers do not widen their own limits.
    median = np.median(satv)
    mad = np.median(np.abs(satv - median))
    robust_sigma = 1.4826 * mad

    distance = n_sigma * robust_sigma
    is_outlier = np.abs(satv - median) > distance

    # The limits are drawn on the original curve, so they follow the season.
    upper = seasonal + median + distance
    lower = seasonal + median - distance

    figure, axes = plt.subplots(figsize=(12, 5))
    axes.plot(series.index, values, linewidth=0.6, color="steelblue",
              label="Average precipitation")
    axes.plot(series.index, upper, linewidth=1.0, color="grey",
              label=f"SPC limits ({n_sigma} sigma)")
    axes.plot(series.index, lower, linewidth=1.0, color="grey")
    axes.scatter(series.index[is_outlier], values[is_outlier],
                 s=14, color="crimson", zorder=3, label="Outliers")

    axes.set_title(
        f"Precipitation with SPC limits (cutoff {cutoff}, {n_sigma} sigma)"
    )
    axes.set_xlabel("Time")
    axes.set_ylabel("Precipitation (mm)")
    axes.grid(alpha=0.3)
    axes.legend(fontsize=8)
    figure.autofmt_xdate()

    outliers = pd.DataFrame({
        "time": series.index[is_outlier],
        column: values[is_outlier],
    })

    print(f"Robust standard deviation: {robust_sigma:.4f}")
    print(f"Outliers: {is_outlier.sum()} of {len(values)} hours "
          f"({100 * is_outlier.mean():.2f} percent)")

    return figure, outliers

In [ ]:
# Default settings.
figure, outliers = precipitation_outliers(weather_no3)
plt.show()

print()
print("The five wettest outliers:")
outliers.nlargest(5, "precipitation")

In [ ]:
# A stricter limit and a smoother seasonal curve give fewer outliers.
figure, strict_outliers = precipitation_outliers(weather_no3, cutoff=10,
                                                 n_sigma=5)
plt.show()

## 6. Anomalies in temperature (LOF)

Local Outlier Factor works differently from the SPC limits above. Instead of
drawing a boundary, it compares how tightly packed each point is against how
tightly packed its neighbours are. A point sitting alone in a thin area gets a
high score, even if its value is not extreme on its own.

The method is given two features: the time and the temperature, both scaled so
neither dominates the distance. That way "unusual" means unusual for that part
of the year, rather than just cold or warm in general.

LOF has no natural threshold, so instead of a limit I tell it what proportion
of the points should be flagged. One percent is the default here.

The points it finds are not the hottest or coldest hours of the year. They are
hours that break away from the hours around them, since each point is compared
against its neighbours in time. That makes it a useful complement to the SPC
limits above, which react to the level instead.

In [ ]:
def temperature_anomalies(weather, proportion=0.01, n_neighbors=20,
                          column="temperature_2m"):
    """Plot temperature and mark anomalies found by Local Outlier Factor.

    proportion is the share of points that should be flagged, so 0.01 marks
    the one percent that look most out of place.

    n_neighbors is how many nearby points each point is compared against. A
    small number reacts to short events, a large one only to longer ones.

    Returns the figure and a table of the anomalies.
    """
    series = weather.set_index("time")[column]
    values = series.to_numpy()

    # Time has to be a number for the distance calculation. Both features are
    # scaled, or the hour count would completely dominate the temperature.
    hours = np.arange(len(values), dtype=float)
    features = np.column_stack([
        (hours - hours.mean()) / hours.std(),
        (values - values.mean()) / values.std(),
    ])

    detector = LocalOutlierFactor(
        n_neighbors=n_neighbors, contamination=proportion
    )
    labels = detector.fit_predict(features)

    # fit_predict marks anomalies with -1. The score says how isolated each
    # point is, where a higher number is more unusual.
    is_anomaly = labels == -1
    score = -detector.negative_outlier_factor_

    figure, axes = plt.subplots(figsize=(12, 5))
    axes.plot(series.index, values, linewidth=0.6, color="steelblue",
              label="Average temperature")
    axes.scatter(series.index[is_anomaly], values[is_anomaly],
                 s=14, color="crimson", zorder=3, label="Anomalies")

    axes.set_title(
        f"Temperature with LOF anomalies ({100 * proportion:.1f} percent flagged)"
    )
    axes.set_xlabel("Time")
    axes.set_ylabel("Temperature (degrees C)")
    axes.grid(alpha=0.3)
    axes.legend(fontsize=8)
    figure.autofmt_xdate()

    anomalies = pd.DataFrame({
        "time": series.index[is_anomaly],
        column: values[is_anomaly],
        "lof_score": score[is_anomaly].round(3),
    })

    print(f"Anomalies: {is_anomaly.sum()} of {len(values)} hours "
          f"({100 * is_anomaly.mean():.2f} percent)")
    print(f"Coldest flagged: {anomalies[column].min():.1f} degrees")
    print(f"Warmest flagged: {anomalies[column].max():.1f} degrees")

    return figure, anomalies

In [ ]:
# One percent flagged, the default.
figure, anomalies = temperature_anomalies(weather_no3)
plt.show()

print()
print("The five most isolated points:")
anomalies.nlargest(5, "lof_score")

In [ ]:
# Raising the proportion flags more, including shorter events.
figure, more_anomalies = temperature_anomalies(weather_no3, proportion=0.03)
plt.show()

## 7. Seasonality in reservoir filling (STL)

STL splits a series into three parts: a trend, a repeating seasonal pattern,
and a remainder of whatever is left over. Reservoir filling is a good case for
it, because the yearly cycle is obvious. Reservoirs fill through the summer
melt and drain through the winter.

The data is weekly, so one season is 52 points. Two settings are left open:

The seasonal smoother decides how much the shape of the yearly cycle is
allowed to change from one year to the next. A small number lets it follow
each year closely, a large number forces one shared shape on all years.

The trend smoother decides how long a movement has to last before it counts as
trend rather than season. The default here is 105 weeks, about two years.

Setting robust to True makes STL downweight single weeks that stick out, so an
unusual year does not bend the trend around itself.

In [ ]:
NVE_DATA_URL = (
    "https://biapi.nve.no/magasinstatistikk/api/Magasinstatistikk/HentOffentligData"
)

# The same English column names the Streamlit app uses, so the two match.
COLUMN_NAMES_EN = {
    "dato_Id": "Date",
    "omrType": "Area type",
    "omrnr": "Area number",
    "fyllingsgrad": "Fill level (fraction)",
    "kapasitet_TWh": "Capacity (TWh)",
    "fylling_TWh": "Stored energy (TWh)",
}


def load_reservoirs():
    """Download the full reservoir statistics from NVE."""
    response = requests.get(NVE_DATA_URL, timeout=60)
    response.raise_for_status()

    data = pd.DataFrame(response.json())
    data = data.rename(columns=COLUMN_NAMES_EN)
    data["Date"] = pd.to_datetime(data["Date"])

    return data.sort_values("Date")


reservoirs = load_reservoirs()
print("Rows:", len(reservoirs))
print("Years:", reservoirs["Date"].dt.year.min(), "to",
      reservoirs["Date"].dt.year.max())
reservoirs.head()

In [ ]:
def reservoir_cycles(reservoirs, area_type="EL", area_number=3,
                     seasonal_smoother=13, trend_smoother=105, robust=False,
                     column="Fill level (fraction)"):
    """Split the reservoir filling of one area into trend, season and remainder.

    seasonal_smoother and trend_smoother are the two window lengths STL uses.
    Both have to be odd numbers. robust tells STL to downweight single weeks
    that stick out instead of letting them pull the trend.

    Returns the figure and a table with the three parts.
    """
    area = reservoirs[(reservoirs["Area type"] == area_type) &
                      (reservoirs["Area number"] == area_number)]
    area = area.sort_values("Date")

    series = area.set_index("Date")[column]

    # A repeated date would break STL, so only the first one is kept.
    series = series[~series.index.duplicated(keep="first")]

    # The data is weekly, so one full season is 52 points.
    result = STL(series, period=52, seasonal=seasonal_smoother,
                 trend=trend_smoother, robust=robust).fit()

    figure, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=True)

    axes[0].plot(series.index, series.to_numpy(), linewidth=0.8,
                 color="steelblue")
    axes[0].set_ylabel("Observed")

    axes[1].plot(series.index, result.trend, linewidth=1.0, color="darkorange")
    axes[1].set_ylabel("Trend")

    axes[2].plot(series.index, result.seasonal, linewidth=0.8, color="seagreen")
    axes[2].set_ylabel("Seasonal")

    axes[3].plot(series.index, result.resid, linewidth=0.6, color="grey")
    axes[3].set_ylabel("Remainder")
    axes[3].set_xlabel("Time")

    for one_axis in axes:
        one_axis.grid(alpha=0.3)

    axes[0].set_title(
        f"STL of {area_type} {area_number} "
        f"(seasonal {seasonal_smoother}, trend {trend_smoother}, robust {robust})"
    )
    figure.autofmt_xdate()
    figure.tight_layout()

    parts = pd.DataFrame({
        "trend": result.trend,
        "seasonal": result.seasonal,
        "remainder": result.resid,
    }, index=series.index)

    # How much of the movement each part explains, as a rough share.
    total = series.var()
    print(f"Weeks in series: {len(series)}")
    print(f"Seasonal share of variance: {result.seasonal.var() / total:.2f}")
    print(f"Trend share of variance: {result.trend.var() / total:.2f}")
    print(f"Remainder standard deviation: {result.resid.std():.4f}")

    return figure, parts

In [ ]:
# Default settings on electricity area 3.
figure, parts = reservoir_cycles(reservoirs)
plt.show()

parts.head()

In [ ]:
# A shorter trend smoother lets more of the slow movement count as trend.
figure, parts_short = reservoir_cycles(reservoirs, trend_smoother=53)
plt.show()

In [ ]:
# The same area as a watercourse region instead, with robust turned on.
figure, parts_vass = reservoir_cycles(reservoirs, area_type="VASS",
                                      area_number=3, robust=True)
plt.show()

## 8. Spectrogram

The STL above assumes one seasonal pattern that holds for the whole period. A
spectrogram does not. It cuts the series into overlapping windows and finds
the frequencies inside each window separately, so a cycle that appears or
disappears partway through shows up as a band that changes brightness.

Frequencies are easier to read as periods, so the vertical axis is the length
of the cycle in weeks instead. The yearly cycle sits at 52 weeks and should be
the brightest band.

Two settings are left open:

The window length is how many weeks go into each slice. A long window
separates close frequencies better but cannot say exactly when something
changed. A short window is the other way around. The default is 208 weeks,
four years.

The overlap is how many weeks two neighbouring windows share. More overlap
gives more windows and a smoother picture, but the extra windows do not add
new information.

In [ ]:
def reservoir_spectrogram(reservoirs, area_type="EL", area_number=3,
                          window_weeks=208, overlap_weeks=156,
                          column="Fill level (fraction)"):
    """Show how the cycles in one area's reservoir filling change over time.

    window_weeks is the length of each slice and overlap_weeks is how much two
    neighbouring slices share. The overlap has to be smaller than the window.

    Returns the figure and a table of what the strongest cycle was in each
    window.
    """
    area = reservoirs[(reservoirs["Area type"] == area_type) &
                      (reservoirs["Area number"] == area_number)]
    area = area.sort_values("Date")

    series = area.set_index("Date")[column]
    series = series[~series.index.duplicated(keep="first")]
    values = series.to_numpy()

    # One sample per week, so fs is 1 and the frequencies come out in cycles
    # per week.
    frequencies, window_centres, power = spectrogram(
        values, fs=1.0, nperseg=window_weeks, noverlap=overlap_weeks
    )

    # The first frequency is zero, which is just the average level and would
    # give a period of infinity. It is dropped before converting.
    frequencies = frequencies[1:]
    power = power[1:, :]
    periods = 1.0 / frequencies

    # scipy counts the window centres in samples, so one sample is seven days.
    times = series.index[0] + pd.to_timedelta(window_centres * 7, unit="D")

    figure, axes = plt.subplots(figsize=(12, 5))

    # Power is drawn in decibels, because the yearly cycle is so much stronger
    # than everything else that a linear scale would show one line only.
    mesh = axes.pcolormesh(times, periods, 10 * np.log10(power),
                           shading="auto", cmap="magma")

    axes.set_yscale("log")
    axes.set_yticks([8, 13, 26, 52, 104, 208])
    axes.set_yticklabels(["8", "13", "26", "52", "104", "208"])
    axes.set_ylabel("Period (weeks)")
    axes.set_xlabel("Time")
    axes.set_title(
        f"Spectrogram of {area_type} {area_number} "
        f"(window {window_weeks} weeks, overlap {overlap_weeks})"
    )
    figure.colorbar(mesh, ax=axes, label="Power (dB)")
    figure.autofmt_xdate()
    figure.tight_layout()

    # The bin closest to one year, so its strength can be followed separately.
    yearly_bin = int(np.argmin(np.abs(periods - 52)))

    rows = []
    for window in range(power.shape[1]):
        strongest = int(np.argmax(power[:, window]))
        rows.append({
            "window_centre": times[window],
            "strongest_period_weeks": round(periods[strongest], 1),
            "yearly_power_db": round(
                10 * np.log10(power[yearly_bin, window]), 1
            ),
        })

    summary = pd.DataFrame(rows)

    print(f"Windows: {power.shape[1]}")
    print(f"Periods covered: {periods.min():.1f} to {periods.max():.1f} weeks")

    return figure, summary

In [ ]:
# Default window of four years.
figure, cycles = reservoir_spectrogram(reservoirs)
plt.show()

cycles.head()

In [ ]:
# A shorter window says more precisely when something changed, but the bands
# get wider because close periods are harder to tell apart.
figure, short_cycles = reservoir_spectrogram(reservoirs, window_weeks=104,
                                             overlap_weeks=78)
plt.show()

In [ ]:
# The strongest cycle in each window, to check that the yearly one dominates.
print("Most common strongest period:")
print(cycles["strongest_period_weeks"].value_counts().head())

## 9. Log

## 10. AI declaration